# Day 34: Long Context Summarisation (Map‑Reduce)

Summarise a full book by splitting, summarising chunks, then combining.

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain.chains.summarize import load_summarize_chain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

## 1. Load a long text (sample: first few chapters of a book)
For demonstration, we'll use a public domain text. You can replace with your own.

In [ ]:
# Download a sample long text (e.g., Alice in Wonderland)
import requests
url = "https://www.gutenberg.org/files/11/11-0.txt"
response = requests.get(url)
long_text = response.text[:50000]  # first 50k chars for speed
print(f"Loaded {len(long_text)} characters")

## 2. Split into chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    length_function=len,
)
chunks = text_splitter.split_text(long_text)
documents = [Document(page_content=chunk) for chunk in chunks]
print(f"Split into {len(documents)} chunks")

## 3. Map‑Reduce summarisation chain

In [ ]:
chain = load_summarize_chain(
    llm,
    chain_type="map_reduce",
    verbose=True  # shows progress
)

summary = chain.invoke(documents)
print("Final summary:\n")
print(summary['output_text'])

## 4. Custom prompt for map and reduce
You can customise the prompts to control output style.

In [ ]:
from langchain.chains.combine_documents.map_reduce import MapReduceDocumentsChain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.prompts import PromptTemplate

map_template = """Write a concise summary of the following text:
{text}
CONCISE SUMMARY:"""
map_prompt = PromptTemplate(template=map_template, input_variables=["text"])

reduce_template = """Combine the following summaries into a final cohesive summary:
{text}
FINAL SUMMARY:"""
reduce_prompt = PromptTemplate(template=reduce_template, input_variables=["text"])

# This is more advanced; for simplicity, stick with default chain for now.